In [2]:
%pip install openai

  Using cached distro-1.9.0-py3-none-any.whl.metadata (6.8 kB)
  Using cached annotated_types-0.7.0-py3-none-any.whl.metadata (15 kB)
  Using cached typing_inspection-0.4.2-py3-none-any.whl.metadata (2.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 3.6 MB/s  0:00:00m eta 0:00:01
Using cached distro-1.9.0-py3-none-any.whl (20 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 2.7 MB/s  0:00:00 eta 0:00:01
Using cached annotated_types-0.7.0-py3-none-any.whl (13 kB)
Using cached typing_inspection-0.4.2-py3-none-any.whl (14 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [openai]2m6/7 [openai]
Note: you may need to restart the kernel to use updated packages.


In [29]:
#CSV Vers.
from openai import OpenAI
from time import sleep
import os
#import glob
import pandas as pd

client = OpenAI(api_key="") #api key here

INPUT_FILE = "baiganyo_combined_master_bul.csv"
INPUT_DIR = "Data/Bulgarian/BaiGanyo/orig"
OUTPUT_DIR = "Data/Bulgarian/BaiGanyo/gpt"
OUTPUT_CSV = "baiganyo_combined_master_gpt.csv"

def translate_text(text, model):
    try:
        response = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": "You are a professional translator，please translate from Bulgarian to English while preserving the Markdown structure and formatting."},
                {"role": "user", "content": f"Please translate the following Bulgarian content：{text}"}
            ],
            temperature=0.3 #lower --> more consistent, literal output
        )
        translated_content = response.choices[0].message.content
        return translated_content.replace("Please translate the following Bulgarian content:", "").strip()
    
    except Exception as e:
        print(f"Translation error: {e}")
        return ""

def translate_csv(model="gpt-5.4-mini"):
    df = pd.read_csv(os.path.join(INPUT_DIR, INPUT_FILE))
    
    translations = []

    print(f"Loaded {len(df)} rows from {INPUT_FILE}")
    print(f"Using model: {model}")

    for i, text in enumerate(df['Content'], 1):
        print(f"Translating row {i}/{len(df)}")

        if pd.isna(text) or not str(text).strip():
            translations.append("")
            continue

        translated_content = translate_text(str(text), model)
        translations.append(translated_content)

        sleep(1)

    df["Translated Content"] = translations
    df.to_csv(os.path.join(OUTPUT_DIR, OUTPUT_CSV), index = False)
    print(f"\n Saved translated CSV: {OUTPUT_CSV}")

    return df


if __name__ == "__main__":

    translate_csv(model="gpt-5.4-mini")


Loaded 915 rows from baiganyo_combined_master_bul.csv
Using model: gpt-5.4-mini
Translating row 1/915
Translating row 2/915
Translating row 3/915
Translating row 4/915
Translating row 5/915
Translating row 6/915
Translating row 7/915
Translating row 8/915
Translating row 9/915
Translating row 10/915
Translating row 11/915
Translating row 12/915
Translating row 13/915
Translating row 14/915
Translating row 15/915
Translating row 16/915
Translating row 17/915
Translating row 18/915
Translating row 19/915
Translating row 20/915
Translating row 21/915


KeyboardInterrupt: 

In [ ]:
#MD vers.
from openai import OpenAI
from time import sleep
import os
import glob

client = OpenAI(api_key="") #api key here

INPUT_DIR = "1tv_news_articles_ge"
OUTPUT_DIR = "1tv_news_articles_gpt"

def translate_text(text, model="gpt-5.4-mini"):
    try:
        response = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": "You are a professional translator，please translate from Bulgarian to English while preserving the Markdown structure and formatting."},
                {"role": "user", "content": f"Please translate the following Bulgarian content：{text}"}
            ],
            temperature=0.3 #lower --> more consistent, literal output
        )
        translated_content = response.choices[0].message.content
        return translated_content.replace("Please translate the following Bulgarian content:", "").strip()
    
    except Exception as e:
        print(f"Translation error: {e}")
        return ""

def translate_single_file(input_file, output_file, model="gpt-5.4-mini"):

    print(f"\n Starting to process files: {os.path.basename(input_file)}")

    with open(input_file, "r", encoding = "utf-8") as f:
        content = f.read()

    if not content.strip():
        print("  File is empty.")
        return

    translated_content = translate_text(content, model)

    with open(output_file, "w", encoding = "utf-8") as f:
        f.write(translated_content)

    print(f"  Saved translated file: {os.path.basename(output_file)}")


def translate_all_files(model="gpt-5.4-mini"):

    #Create output directory if it doesn't exist
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    
    # Find all files
    pattern = os.path.join(INPUT_DIR, "*.md")
    files = glob.glob(pattern)
    
    if not files:
        print(f"No file found！Please check directory: {INPUT_DIR}")
        return
    
    print(f"Found {len(files)} files to translate")
    print(f"Using model: {model}")
    
    for i, input_file in enumerate(files, 1):
        print(f"\n========== Processing {i}/{len(files)} files ==========")
        
        # Create file names
        base_name = os.path.basename(input_file)
        output_name = base_name.replace('.md', f'_{model}_translated.md')
        output_file = os.path.join(OUTPUT_DIR, output_name)
        
        # Translate file
        translate_single_file(input_file, output_file, model)

        sleep(1)
        
    print(f"\n All files translated！Saved at: {OUTPUT_DIR}")


if __name__ == "__main__":

    translate_all_files(model="gpt-5.4-mini")


In [13]:
%pip install -U google-genai

  Using cached tenacity-9.1.4-py3-none-any.whl.metadata (1.2 kB)
  Using cached pyasn1_modules-0.4.2-py3-none-any.whl.metadata (3.5 kB)
  Using cached cffi-2.0.0-cp310-cp310-macosx_11_0_arm64.whl.metadata (2.6 kB)
  Using cached pyasn1-0.6.3-py3-none-any.whl.metadata (8.4 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 791.4/791.4 kB 1.8 MB/s  0:00:0036m-:--:--
Using cached tenacity-9.1.4-py3-none-any.whl (28 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 970.3 kB/s  0:00:08m0:00:0100:01
Using cached cffi-2.0.0-cp310-cp310-macosx_11_0_arm64.whl (180 kB)
Using cached pyasn1_modules-0.4.2-py3-none-any.whl (181 kB)
Using cached pyasn1-0.6.3-py3-none-any.whl (83 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9/9 [google-genai] [google-genai]
Note: you may need to restart the kernel to use updated packages.


In [30]:
#CSV vers.
from google import genai
from google.genai import types
from time import sleep
import os
#import glob
import pandas as pd

client = genai.Client(api_key="") #api key here 

INPUT_FILE = "baiganyo_combined_master_bul.csv"
INPUT_DIR = "Data/Bulgarian/BaiGanyo/orig"
OUTPUT_DIR = "Data/Bulgarian/BaiGanyo/gem"
OUTPUT_CSV = "baiganyo_combined_master_gem.csv"

def translate_text(text, model="gemini-3-flash-preview"):
    try:
        response = client.models.generate_content(
            model=model,
            contents=f"Please translate the following Bulgarian content：{text}",
            config=types.GenerateContentConfig(
                temperature=0.3,
                system_instruction="You are a professional translator，please translate from Bulgarian to English while preserving the Markdown structure and formatting. Return only the translation.",
            )
        )
        translated_content = response.text if response.text else ""
        return translated_content.replace("Please translate the following Bulgarian content:", "").strip()
    
    except Exception as e:
        print(f"Translation error: {e}")
        return ""

def translate_csv(model):
    df = pd.read_csv(os.path.join(INPUT_DIR, INPUT_FILE))
    
    translations = []

    print(f"Loaded {len(df)} rows from {INPUT_FILE}")
    print(f"Using model: {model}")

    for i, text in enumerate(df['Content'], 1):
        print(f"Translating row {i}/{len(df)}")

        if pd.isna(text) or not str(text).strip():
            translations.append("")
            continue

        translated_content = translate_text(str(text), model)
        translations.append(translated_content)

        sleep(1)

    df["Translated Content"] = translations
    df.to_csv(os.path.join(OUTPUT_DIR, OUTPUT_CSV), index = False)
    print(f"\n Saved translated CSV: {OUTPUT_CSV}")

    return df


if __name__ == "__main__":

    translate_csv(model="gemini-3-flash-preview")


Loaded 915 rows from baiganyo_combined_master_bul.csv
Using model: gemini-3-flash-preview
Translating row 1/915
Translating row 2/915
Translating row 3/915
Translating row 4/915
Translating row 5/915
Translating row 6/915
Translating row 7/915
Translating row 8/915
Translating row 9/915
Translating row 10/915
Translating row 11/915
Translating row 12/915
Translating row 13/915
Translating row 14/915
Translating row 15/915
Translating row 16/915
Translating row 17/915
Translating row 18/915
Translating row 19/915
Translating row 20/915
Translating row 21/915
Translating row 22/915
Translating row 23/915
Translating row 24/915
Translating row 25/915
Translating row 26/915
Translating row 27/915
Translating row 28/915
Translating row 29/915
Translating row 30/915
Translating row 31/915
Translating row 32/915
Translating row 33/915
Translating row 34/915
Translating row 35/915
Translating row 36/915
Translating row 37/915
Translating row 38/915
Translating row 39/915
Translating row 40/915

KeyboardInterrupt: 

In [16]:
#MD vers.
from google import genai
from google.genai import types
from time import sleep
import os
import glob

client = genai.Client(api_key="") #api key here 

INPUT_DIR = "1tv_news_articles_ge"
OUTPUT_DIR = "1tv_news_articles_gem"

def translate_text(text, model="gemini-3-flash-preview"):
    try:
        response = client.models.generate_content(
            model=model,
            contents=f"Please translate the following Georgian content：{text}",
            config=types.GenerateContentConfig(
                temperature=0.3,
                system_instruction="You are a professional translator，please translate from Georgian to English while preserving the Markdown structure and formatting.",
            )
        )
        translated_content = response.text if response.text else ""
        return translated_content.replace("Please translate the following Georgian content:", "").strip()
    
    except Exception as e:
        print(f"Translation error: {e}")
        return ""


def translate_single_file(input_file, output_file, model="gemini-3-flash-preview"):

    print(f"\n Starting to process files: {os.path.basename(input_file)}")

    with open(input_file, "r", encoding = "utf-8") as f:
        content = f.read()

    if not content.strip():
        print("  File is empty.")
        return

    translated_content = translate_text(content, model)

    with open(output_file, "w", encoding = "utf-8") as f:
        f.write(translated_content)

    print(f"  Saved translated file: {os.path.basename(output_file)}")


def translate_all_files(model="gemini-3-flash-preview"):

    #Create output directory if it doesn't exist
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    
    # Find all files
    pattern = os.path.join(INPUT_DIR, "*.md")
    files = glob.glob(pattern)
    
    if not files:
        print(f"No file found！Please check directory: {INPUT_DIR}")
        return
    
    print(f"Found {len(files)} files to translate")
    print(f"Using model: {model}")
    
    for i, input_file in enumerate(files, 1):
        print(f"\n========== Processing {i}/{len(files)} files ==========")
        
        # Create file names
        base_name = os.path.basename(input_file)
        output_name = base_name.replace('.md', f'_{model}_translated.md')
        output_file = os.path.join(OUTPUT_DIR, output_name)
        
        # Translate file
        translate_single_file(input_file, output_file, model)

        sleep(1)
        
    print(f"\n All files translated！Saved at: {OUTPUT_DIR}")


if __name__ == "__main__":

    translate_all_files(model="gemini-3-flash-preview")

Found 10 files to translate
Using model: gemini-3-flash-preview

========== Processing 1/10 files ==========

 Starting to process files: tbilisi_offers_free_metro.md
  Saved translated file: tbilisi_offers_free_metro_gemini-3-flash-preview_translated.md

========== Processing 2/10 files ==========

 Starting to process files: deputy_economy_minister_10_04_26.md
  Saved translated file: deputy_economy_minister_10_04_26_gemini-3-flash-preview_translated.md

========== Processing 3/10 files ==========

 Starting to process files: vespers_of_the_descent.md
  Saved translated file: vespers_of_the_descent_gemini-3-flash-preview_translated.md

========== Processing 4/10 files ==========

 Starting to process files: good_friday.md
  Saved translated file: good_friday_gemini-3-flash-preview_translated.md

========== Processing 5/10 files ==========

 Starting to process files: metropolitan_shio.md
  Saved translated file: metropolitan_shio_gemini-3-flash-preview_translated.md

========== Proce

In [22]:
%pip install --upgrade google-cloud-translate

  Using cached google_cloud_translate-3.26.0-py3-none-any.whl.metadata (9.7 kB)
  Using cached google_api_core-2.30.3-py3-none-any.whl.metadata (3.1 kB)
  Using cached grpc_google_iam_v1-0.14.4-py3-none-any.whl.metadata (9.1 kB)
  Using cached grpcio_status-1.80.0-py3-none-any.whl.metadata (1.3 kB)
  Using cached protobuf-6.33.6-cp39-abi3-macosx_10_9_universal2.whl.metadata (593 bytes)
Using cached google_cloud_translate-3.26.0-py3-none-any.whl (210 kB)
Using cached google_api_core-2.30.3-py3-none-any.whl (173 kB)
Using cached grpc_google_iam_v1-0.14.4-py3-none-any.whl (32 kB)
Using cached grpcio_status-1.80.0-py3-none-any.whl (14 kB)
Using cached protobuf-6.33.6-cp39-abi3-macosx_10_9_universal2.whl (427 kB)
  Attempting uninstall: protobuf
    Found existing installation: protobuf 7.34.1
    Uninstalling protobuf-7.34.1:
      Successfully uninstalled protobuf-7.34.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8/8 [google-cloud-translate]e-cloud-translate]
Note: you may need to restar

In [28]:
#CSV vers.
import os
import glob
from time import sleep
from google.cloud import translate_v3 as translate

os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = ""

PROJECT_ID = ""
LOCATION = "global"  # 'global' is standard unless you use specific regional endpoints

INPUT_FILE = "baiganyo_combined_master_bul.csv"
INPUT_DIR = "Data/Bulgarian/BaiGanyo/orig"
OUTPUT_DIR = "Data/Bulgarian/BaiGanyo/ggl"
OUTPUT_CSV = "baiganyo_combined_master_ggl.csv"

client = translate.TranslationServiceClient()
parent = f"projects/{PROJECT_ID}/locations/{LOCATION}"

def translate_text(text, target_lang="en", source_lang="bg"):
    """Translates text using the Advanced (v3) API."""
    try:
        # V3 requires a list of strings, even for one translation
        response = client.translate_text(
            request={
                "parent": parent,
                "contents": [text],
                "mime_type": "text/plain",  # Use "text/html" if you have complex Markdown
                "source_language_code": source_lang,
                "target_language_code": target_lang,
            }
        )

        # Extract the translation from the response object
        return response.translations[0].translated_text
    
    except Exception as e:
        print(f"Translation error: {e}")
        return ""

def translate_csv():
    df = pd.read_csv(os.path.join(INPUT_DIR, INPUT_FILE))
    
    translations = []

    print(f"Loaded {len(df)} rows from {INPUT_FILE}")

    for i, text in enumerate(df['Content'], 1):
        print(f"Translating row {i}/{len(df)}")

        if pd.isna(text) or not str(text).strip():
            translations.append("")
            continue

        translated_content = translate_text(str(text))
        translations.append(translated_content)

        sleep(1)

    df["Translated Content"] = translations
    df.to_csv(os.path.join(OUTPUT_DIR, OUTPUT_CSV), index = False)
    print(f"\n Saved translated CSV: {OUTPUT_CSV}")

    return df

if __name__ == "__main__":

    translate_csv()


Loaded 915 rows from baiganyo_combined_master_bul.csv
Translating row 1/915
Translating row 2/915
Translating row 3/915
Translating row 4/915
Translating row 5/915
Translating row 6/915
Translating row 7/915
Translating row 8/915
Translating row 9/915
Translating row 10/915
Translating row 11/915
Translating row 12/915
Translating row 13/915
Translating row 14/915
Translating row 15/915
Translating row 16/915
Translating row 17/915
Translating row 18/915
Translating row 19/915
Translating row 20/915
Translating row 21/915
Translating row 22/915
Translating row 23/915
Translating row 24/915
Translating row 25/915
Translating row 26/915
Translating row 27/915
Translating row 28/915
Translating row 29/915
Translating row 30/915
Translating row 31/915
Translating row 32/915
Translating row 33/915
Translating row 34/915
Translating row 35/915
Translating row 36/915
Translating row 37/915
Translating row 38/915
Translating row 39/915
Translating row 40/915
Translating row 41/915
Translating 

In [11]:
#MD Vers.
import os
import glob
from time import sleep
from google.cloud import translate_v3 as translate

os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = "" #key

PROJECT_ID = ""
LOCATION = "global"  # 'global' is standard unless you use specific regional endpoints

INPUT_DIR = "1tv_news_articles_ge"
OUTPUT_DIR = "1tv_news_articles_ggl"

client = translate.TranslationServiceClient()
parent = f"projects/{PROJECT_ID}/locations/{LOCATION}"

def translate_text(text, target_lang="en", source_lang="ka"):
    """Translates text using the Advanced (v3) API."""
    try:
        # V3 requires a list of strings, even for one translation
        response = client.translate_text(
            request={
                "parent": parent,
                "contents": [text],
                "mime_type": "text/plain",  # Use "text/html" if you have complex Markdown
                "source_language_code": source_lang,
                "target_language_code": target_lang,
            }
        )

        # Extract the translation from the response object
        return response.translations[0].translated_text
    
    except Exception as e:
        print(f"Translation error: {e}")
        return ""

def translate_single_file(input_file, output_file):
    print(f"\n Starting to process: {os.path.basename(input_file)}")

    with open(input_file, "r", encoding="utf-8") as f:
        content = f.read()

    if not content.strip():
        print("  File is empty.")
        return

    translated_content = translate_text(content)

    if translated_content:
        with open(output_file, "w", encoding="utf-8") as f:
            f.write(translated_content)
        print(f"  Saved translated file: {os.path.basename(output_file)}")
    else:
        print("  Failed to translate.")

def translate_all_files():
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    pattern = os.path.join(INPUT_DIR, "*.md")
    files = glob.glob(pattern)
    
    if not files:
        print(f"No files found in {INPUT_DIR}")
        return
    
    print(f"Found {len(files)} files. Using Cloud Translation Advanced (v3)...")
    
    for i, input_file in enumerate(files, 1):
        print(f"\n========== {i}/{len(files)} ==========")
        base_name = os.path.basename(input_file)
        output_name = base_name.replace('.md', '_ggl_translated.md')
        output_file = os.path.join(OUTPUT_DIR, output_name)
        
        translate_single_file(input_file, output_file)
        sleep(0.1)

if __name__ == "__main__":
    translate_all_files()

Found 10 files. Using Cloud Translation Advanced (v3)...

========== 1/10 ==========

 Starting to process: tbilisi_offers_free_metro.md
  Saved translated file: tbilisi_offers_free_metro_ggl_translated.md

========== 2/10 ==========

 Starting to process: deputy_economy_minister_10_04_26.md
  Saved translated file: deputy_economy_minister_10_04_26_ggl_translated.md

========== 3/10 ==========

 Starting to process: vespers_of_the_descent.md
  Saved translated file: vespers_of_the_descent_ggl_translated.md

========== 4/10 ==========

 Starting to process: good_friday.md
  Saved translated file: good_friday_ggl_translated.md

========== 5/10 ==========

 Starting to process: metropolitan_shio.md
  Saved translated file: metropolitan_shio_ggl_translated.md

========== 6/10 ==========

 Starting to process: georgia_to_recognise.md
  Saved translated file: georgia_to_recognise_ggl_translated.md

========== 7/10 ==========

 Starting to process: people's_power's.md
  Saved translated file: